# 01 Dataset Overview

Repository notebook for auditing prepared mammography manifests, cohort structure, labels, view availability, bilateral availability, and temporal availability.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "src").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

def read_optional_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported table format: {path.suffix}")

def find_files(patterns, roots):
    rows = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.rglob(pattern):
                rows.append({
                    "file": path.name,
                    "path": str(path.relative_to(PROJECT_ROOT)) if PROJECT_ROOT in path.parents else str(path),
                    "size_kb": round(path.stat().st_size / 1024, 2),
                    "modified": pd.to_datetime(path.stat().st_mtime, unit="s"),
                })
    return pd.DataFrame(rows).sort_values("modified", ascending=False).reset_index(drop=True) if rows else pd.DataFrame()

## Locate manifest and dataset files

The notebook searches standard repository folders for prepared manifests and metadata tables.

In [ ]:
candidate_roots = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "outputs",
    PROJECT_ROOT / "results",
]

manifest_files = find_files(
    patterns=["*manifest*.csv", "*metadata*.csv", "*cohort*.csv", "*audit*.csv"],
    roots=candidate_roots,
)

manifest_files

## Load primary manifest

Set `MANIFEST_PATH` manually if the desired file is not the first detected manifest.

In [ ]:
MANIFEST_PATH = None

if MANIFEST_PATH is None and not manifest_files.empty:
    MANIFEST_PATH = PROJECT_ROOT / manifest_files.iloc[0]["path"]

manifest_df = read_optional_table(Path(MANIFEST_PATH)) if MANIFEST_PATH is not None else pd.DataFrame()
print(f"Loaded manifest shape: {manifest_df.shape}")
manifest_df.head()

## Dataset audit summary

In [ ]:
if manifest_df.empty:
    audit_summary = pd.DataFrame()
else:
    audit_summary = pd.DataFrame([
        {"metric": "n_rows", "value": len(manifest_df)},
        {"metric": "n_columns", "value": manifest_df.shape[1]},
        {"metric": "missing_values", "value": int(manifest_df.isna().sum().sum())},
        {"metric": "duplicate_rows", "value": int(manifest_df.duplicated().sum())},
    ])
audit_summary

## Patient, study, and label distribution

In [ ]:
id_candidates = ["patient_id", "study_id", "image_id", "exam_id"]
label_candidates = ["diagnosis_label", "label", "target", "cancer", "malignant"]

summary_rows = []

for col in id_candidates:
    if col in manifest_df.columns:
        summary_rows.append({"field": col, "unique_count": manifest_df[col].nunique(), "missing": manifest_df[col].isna().sum()})

id_summary = pd.DataFrame(summary_rows)
id_summary

In [ ]:
label_col = next((c for c in label_candidates if c in manifest_df.columns), None)

if label_col is not None:
    label_distribution = (
        manifest_df[label_col]
        .value_counts(dropna=False)
        .rename_axis(label_col)
        .reset_index(name="count")
    )
    label_distribution["fraction"] = label_distribution["count"] / label_distribution["count"].sum()
else:
    label_distribution = pd.DataFrame()

label_distribution

## View and pathway availability audit

In [ ]:
availability_cols = [
    c for c in manifest_df.columns
    if any(token in c.lower() for token in ["available", "mask", "complete", "temporal_case", "has_prior"])
]

availability_summary = []

for col in availability_cols:
    values = pd.to_numeric(manifest_df[col], errors="coerce")
    availability_summary.append({
        "column": col,
        "mean": values.mean(),
        "active_count": int((values == 1).sum()) if values.notna().any() else None,
        "missing": int(manifest_df[col].isna().sum()),
    })

pd.DataFrame(availability_summary).sort_values("column") if availability_summary else pd.DataFrame()

## Split audit

In [ ]:
split_col = "split" if "split" in manifest_df.columns else None

if split_col is not None:
    split_summary = (
        manifest_df.groupby(split_col)
        .agg(n_records=(split_col, "size"))
        .reset_index()
    )
    if "patient_id" in manifest_df.columns:
        patient_counts = manifest_df.groupby(split_col)["patient_id"].nunique().reset_index(name="n_patients")
        split_summary = split_summary.merge(patient_counts, on=split_col, how="left")
    if label_col is not None:
        pos = manifest_df.groupby(split_col)[label_col].mean().reset_index(name="positive_fraction")
        split_summary = split_summary.merge(pos, on=split_col, how="left")
else:
    split_summary = pd.DataFrame()

split_summary